In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

file_path = Path("../data/online_retail_II.xlsx")

print("File exists:", file_path.exists())

File exists: True


In [3]:
raw_1 = pd.read_excel(
    file_path, sheet_name="Year 2009-2010"
)

raw_2 = pd.read_excel(
    file_path, sheet_name="Year 2010-2011"
)

print(raw_1.shape)
print(raw_2.shape)

(525461, 8)
(541910, 8)


In [4]:
df_clean = pd.concat(
    [raw_1, raw_2],
    ignore_index=True
)

print("Combined shape:", df_clean.shape)
print("Exact duplicate rows:", df_clean.duplicated().sum())

Combined shape: (1067371, 8)
Exact duplicate rows: 34335


In [5]:
df_clean.columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country"
]

print(df_clean.columns.tolist())

['invoice', 'stock_code', 'description', 'quantity', 'invoice_date', 'price', 'customer_id', 'country']


In [6]:
text_columns = [
    "invoice", "stock_code", "description", "country"
]

for column in text_columns:
    df_clean[column] = (
        df_clean[column].astype("string").str.strip()
    )

print(df_clean[text_columns].head())

  invoice stock_code                          description         country
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS  United Kingdom
1  489434     79323P                   PINK CHERRY LIGHTS  United Kingdom
2  489434     79323W                  WHITE CHERRY LIGHTS  United Kingdom
3  489434      22041          RECORD FRAME 7" SINGLE SIZE  United Kingdom
4  489434      21232       STRAWBERRY CERAMIC TRINKET BOX  United Kingdom


In [7]:
df_clean["invoice_date"] = pd.to_datetime(
    df_clean["invoice_date"], errors="coerce"
)

df_clean["quantity"] = pd.to_numeric(
    df_clean["quantity"], errors="coerce"
)

df_clean["price"] = pd.to_numeric(
    df_clean["price"], errors="coerce"
)

print(df_clean.dtypes)
print("\nMissing dates:", df_clean["invoice_date"].isna().sum())
print("Missing quantities:", df_clean["quantity"].isna().sum())
print("Missing prices:", df_clean["price"].isna().sum())

invoice                 string
stock_code              string
description             string
quantity                 int64
invoice_date    datetime64[us]
price                  float64
customer_id            float64
country                 string
dtype: object

Missing dates: 0
Missing quantities: 0
Missing prices: 0


In [8]:
df_clean["line_value"] = (
    df_clean["quantity"] * df_clean["price"]
)

df_clean["is_cancellation_invoice"] = (
    df_clean["invoice"].str.startswith("C", na=False)
)

df_clean["missing_customer_id"] = (
    df_clean["customer_id"].isna()
)

df_clean["negative_quantity"] = (
    df_clean["quantity"] < 0
)

df_clean["zero_price"] = (
    df_clean["price"] == 0
)

df_clean["negative_price"] = (
    df_clean["price"] < 0
)

print(df_clean[
    ["invoice", "quantity", "price", "line_value",
     "is_cancellation_invoice", "missing_customer_id"]
].head())

  invoice  quantity  price  line_value  is_cancellation_invoice  \
0  489434        12   6.95        83.4                    False   
1  489434        12   6.75        81.0                    False   
2  489434        12   6.75        81.0                    False   
3  489434        48   2.10       100.8                    False   
4  489434        24   1.25        30.0                    False   

   missing_customer_id  
0                False  
1                False  
2                False  
3                False  
4                False  


In [9]:
df_clean["is_in_duplicate_group"] = (
    df_clean.duplicated(keep=False)
)

df_clean["is_duplicate_extra"] = (
    df_clean.duplicated(keep="first")
)

print(
    "Rows in duplicate groups:",
    df_clean["is_in_duplicate_group"].sum()
)

print(
    "Extra duplicate occurrences:",
    df_clean["is_duplicate_extra"].sum()
)

Rows in duplicate groups: 67242
Extra duplicate occurrences: 34335


In [10]:
conditions = [
    df_clean["stock_code"].eq("B"),
    df_clean["is_cancellation_invoice"],
    df_clean["negative_quantity"]
]

labels = [
    "bad_debt_adjustment",
    "cancellation_style_invoice",
    "negative_quantity_non_c_invoice"
]

df_clean["transaction_category"] = np.select(
    conditions,
    labels,
    default="other_or_ordinary"
)

print(
    df_clean["transaction_category"]
    .value_counts(dropna=False)
)

print(
    df_clean.groupby("transaction_category")["line_value"]
    .agg(["count", "sum"])
    .round(2)
)

transaction_category
other_or_ordinary                  1044414
cancellation_style_invoice           19494
negative_quantity_non_c_invoice       3457
bad_debt_adjustment                      6
Name: count, dtype: int64
                                   count          sum
transaction_category                                 
bad_debt_adjustment                    6   -147614.08
cancellation_style_invoice         19494  -1526667.86
negative_quantity_non_c_invoice     3457         0.00
other_or_ordinary                1044414  20961532.51


In [11]:
print("Original combined rows:", len(raw_1) + len(raw_2))
print("Prepared rows:", len(df_clean))
print("Rows lost:", len(raw_1) + len(raw_2) - len(df_clean))

print("Duplicate extras:", df_clean["is_duplicate_extra"].sum())
print("Missing Customer IDs:", df_clean["customer_id"].isna().sum())
print("Missing descriptions:", df_clean["description"].isna().sum())

Original combined rows: 1067371
Prepared rows: 1067371
Rows lost: 0
Duplicate extras: 34335
Missing Customer IDs: 243007
Missing descriptions: 4382


In [12]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "online_retail_prepared.csv"

df_clean.to_csv(output_file, index=False)

print("Saved:", output_file.resolve())
print("File size (MB):", round(output_file.stat().st_size / 1_000_000, 2))

Saved: D:\online-retail-analytics\data\processed\online_retail_prepared.csv
File size (MB): 167.28


In [13]:
print("Combined line value:", round(df_clean["line_value"].sum(), 2))

print("Category line values:")
print(
    df_clean.groupby("transaction_category")["line_value"]
    .sum()
    .round(2)
)

print(
    "Category total:",
    round(
        df_clean.groupby("transaction_category")["line_value"]
        .sum()
        .sum(),
        2
    )
)

Combined line value: 19287250.57
Category line values:
transaction_category
bad_debt_adjustment                 -147614.08
cancellation_style_invoice         -1526667.86
negative_quantity_non_c_invoice           0.00
other_or_ordinary                  20961532.51
Name: line_value, dtype: float64
Category total: 19287250.57


In [14]:
mask = (
    (df_clean["quantity"] < 0)
    & ~df_clean["is_cancellation_invoice"]
    & (df_clean["stock_code"] != "B")
)

print("Rows:", mask.sum())
print("Line value:", df_clean.loc[mask, "line_value"].sum())
print("Price distribution:")
print(df_clean.loc[mask, "price"].describe())

print(
    df_clean.loc[
        mask,
        ["invoice", "stock_code", "description",
         "quantity", "price", "customer_id"]
    ].head(20).to_string(index=False)
)

Rows: 3457
Line value: 0.0
Price distribution:
count    3457.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: price, dtype: float64
invoice stock_code     description  quantity  price  customer_id
 489464      21733    85123a mixed       -96    0.0          NaN
 489463      71477           short      -240    0.0          NaN
 489467     85123A     21733 mixed      -192    0.0          NaN
 489521      21646            <NA>       -50    0.0          NaN
 489655      20683            <NA>       -44    0.0          NaN
 489660      35956            lost     -1043    0.0          NaN
 489663     35605A         damages      -117    0.0          NaN
 489806      18010            <NA>      -770    0.0          NaN
 489820      21133 invcd as 84879?      -720    0.0          NaN
 489821     85049G            <NA>      -240    0.0          NaN
 489899    79323GR    sold as gold      -954    0.0          NaN
 489901      2109

In [15]:
duplicate_mask = df_clean["is_duplicate_extra"]

print("Extra duplicate rows:", duplicate_mask.sum())
print(
    "Their combined line value:",
    round(df_clean.loc[duplicate_mask, "line_value"].sum(), 2)
)
print(
    "Their absolute line value:",
    round(df_clean.loc[duplicate_mask, "line_value"].abs().sum(), 2)
)

Extra duplicate rows: 34335
Their combined line value: 431716.87
Their absolute line value: 560951.37


In [16]:
duplicate_rows = df_clean[df_clean["is_duplicate_extra"]]

print("Invoices containing extra duplicate rows:",
      duplicate_rows["invoice"].nunique())

print(
    duplicate_rows.groupby("invoice")
    .size()
    .sort_values(ascending=False)
    .head(10)
)

Invoices containing extra duplicate rows: 5391
invoice
537434    675
538071    652
537638    601
537237    597
536876    593
536592    592
537823    591
537240    568
537666    536
536544    527
dtype: int64


In [17]:
missing_id = df_clean["customer_id"].isna()

print("Missing-ID rows:", missing_id.sum())
print(
    "Missing-ID line value:",
    round(df_clean.loc[missing_id, "line_value"].sum(), 2)
)
print(
    "Known-ID line value:",
    round(df_clean.loc[~missing_id, "line_value"].sum(), 2)
)

Missing-ID rows: 243007
Missing-ID line value: 2638958.18
Known-ID line value: 16648292.39


In [18]:
for label, mask in [
    ("Zero price", df_clean["price"] == 0),
    ("Negative price", df_clean["price"] < 0),
    ("Negative quantity", df_clean["quantity"] < 0)
]:
    print(
        label,
        "| rows:", int(mask.sum()),
        "| line value:", round(df_clean.loc[mask, "line_value"].sum(), 2)
    )

Zero price | rows: 6202 | line value: 0.0
Negative price | rows: 5 | line value: -158676.14
Negative quantity | rows: 22950 | line value: -1527041.43


In [19]:
missing_desc = df_clean["description"].isna()

print("Missing-description rows:", missing_desc.sum())
print(
    df_clean.loc[missing_desc, "transaction_category"]
    .value_counts(dropna=False)
)

print(
    df_clean.loc[
        missing_desc,
        ["invoice", "stock_code", "quantity", "price", "customer_id"]
    ].head(10).to_string(index=False)
)

Missing-description rows: 4382
transaction_category
negative_quantity_non_c_invoice    2689
other_or_ordinary                  1693
Name: count, dtype: int64
invoice stock_code  quantity  price  customer_id
 489521      21646       -50    0.0          NaN
 489655      20683       -44    0.0          NaN
 489659      21350       230    0.0          NaN
 489781      84292        17    0.0          NaN
 489806      18010      -770    0.0          NaN
 489821     85049G      -240    0.0          NaN
 489882     35751C        12    0.0          NaN
 489898     79323G       954    0.0          NaN
 489901      21098      -200    0.0          NaN
 489903      21166        48    0.0          NaN


In [20]:
adjustments = df_clean[
    df_clean["invoice"].str.startswith("A", na=False)
]

print("Adjustment-style rows:", len(adjustments))
print("Distinct adjustment invoice IDs:",
      adjustments["invoice"].nunique())

print(
    adjustments[
        ["invoice", "stock_code", "description",
         "quantity", "price", "line_value"]
    ].to_string(index=False)
)

Adjustment-style rows: 6
Distinct adjustment invoice IDs: 6
invoice stock_code     description  quantity     price  line_value
A506401          B Adjust bad debt         1 -53594.36   -53594.36
A516228          B Adjust bad debt         1 -44031.79   -44031.79
A528059          B Adjust bad debt         1 -38925.87   -38925.87
A563185          B Adjust bad debt         1  11062.06    11062.06
A563186          B Adjust bad debt         1 -11062.06   -11062.06
A563187          B Adjust bad debt         1 -11062.06   -11062.06


In [21]:
df_clean["review_status"] = np.select(
    [
        df_clean["stock_code"].eq("B"),
        df_clean["is_cancellation_invoice"],
        df_clean["negative_quantity"],
        df_clean["negative_price"],
        df_clean["is_duplicate_extra"],
        df_clean["missing_customer_id"],
        df_clean["description"].isna()
    ],
    [
        "Review bad-debt adjustment",
        "Review cancellation-style invoice",
        "Review negative quantity",
        "Review negative price",
        "Review duplicate occurrence",
        "Customer ID missing",
        "Description missing"
    ],
    default="No flagged condition"
)

print(df_clean["review_status"].value_counts())

review_status
No flagged condition                 779495
Customer ID missing                  231038
Review duplicate occurrence           33881
Review cancellation-style invoice     19494
Review negative quantity               3457
Review bad-debt adjustment                6
Name: count, dtype: int64


In [22]:
output_file = Path(
    "../data/processed/online_retail_prepared.csv"
)

df_clean.to_csv(output_file, index=False)

print("Saved:", output_file.resolve())
print("Rows:", len(df_clean))
print("Columns:", len(df_clean.columns))
print("Rows lost:", 1067371 - len(df_clean))

Saved: D:\online-retail-analytics\data\processed\online_retail_prepared.csv
Rows: 1067371
Columns: 18
Rows lost: 0


In [28]:

from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent
xlsx_path = project_dir / "data" / "online_retail_II.xlsx"

sheet1 = pd.read_excel(
    xlsx_path, sheet_name="Year 2009-2010"
)
sheet2 = pd.read_excel(
    xlsx_path, sheet_name="Year 2010-2011"
)

date_col = "InvoiceDate"
start = pd.Timestamp("2010-12-01")
end = pd.Timestamp("2010-12-09 23:59:59")

overlap1 = sheet1[
    pd.to_datetime(sheet1[date_col]).between(start, end)
].copy()
overlap2 = sheet2[
    pd.to_datetime(sheet2[date_col]).between(start, end)
].copy()

columns = list(sheet1.columns)

print("Sheet 1 overlap rows:", len(overlap1))
print("Sheet 2 overlap rows:", len(overlap2))
print("Sheet 1 duplicate occurrences within overlap:",
      int(overlap1.duplicated(subset=columns).sum()))
print("Sheet 2 duplicate occurrences within overlap:",
      int(overlap2.duplicated(subset=columns).sum()))

# Compare distinct row patterns across worksheets.
a = overlap1.drop_duplicates(subset=columns)
b = overlap2.drop_duplicates(subset=columns)

shared = a.merge(b, on=columns, how="inner")

print("Distinct matching row patterns across sheets:", len(shared))


Sheet 1 overlap rows: 22523
Sheet 2 overlap rows: 22523
Sheet 1 duplicate occurrences within overlap: 321
Sheet 2 duplicate occurrences within overlap: 321
Distinct matching row patterns across sheets: 22202


In [29]:
print("Sheet 1 overlap rows:", len(overlap1))
print("Sheet 2 overlap rows:", len(overlap2))

print(
    "Unique exact rows in sheet 1:",
    len(overlap1.drop_duplicates())
)
print(
    "Unique exact rows in sheet 2:",
    len(overlap2.drop_duplicates())
)

print(
    "Duplicate occurrences within sheet 1 overlap:",
    int(overlap1.duplicated().sum())
)
print(
    "Duplicate occurrences within sheet 2 overlap:",
    int(overlap2.duplicated().sum())
)

Sheet 1 overlap rows: 22523
Sheet 2 overlap rows: 22523
Unique exact rows in sheet 1: 22202
Unique exact rows in sheet 2: 22202
Duplicate occurrences within sheet 1 overlap: 321
Duplicate occurrences within sheet 2 overlap: 321


In [30]:
columns = list(sheet1.columns)

a = overlap1.drop_duplicates(subset=columns)
b = overlap2.drop_duplicates(subset=columns)

matched = a.merge(b, on=columns, how="inner")

print("Distinct rows in sheet 1:", len(a))
print("Distinct rows in sheet 2:", len(b))
print("Distinct row patterns shared across sheets:", len(matched))

Distinct rows in sheet 1: 22202
Distinct rows in sheet 2: 22202
Distinct row patterns shared across sheets: 22202


In [32]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent
csv_path = project_dir / "data" / "processed" / "online_retail_prepared.csv"

print("CSV exists:", csv_path.exists())

df = pd.read_csv(
    csv_path,
    parse_dates=["invoice_date"],
    dtype={
        "invoice": "string",
        "stock_code": "string",
        "description": "string",
        "country": "string",
        "transaction_category": "string",
    },
)

print("Prepared dataset shape:", df.shape)
print("Date range:", df["invoice_date"].min(), "to", df["invoice_date"].max())

CSV exists: True
Prepared dataset shape: (1067371, 18)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


In [33]:
cutoff = pd.Timestamp("2010-12-10")

raw_2["InvoiceDate"] = pd.to_datetime(raw_2["InvoiceDate"])
raw_1["InvoiceDate"] = pd.to_datetime(raw_1["InvoiceDate"])

raw_2_nonoverlap = raw_2[
    raw_2["InvoiceDate"] >= cutoff
].copy()

combined = pd.concat(
    [raw_1, raw_2_nonoverlap],
    ignore_index=True
)

print("Sheet 1 rows:", len(raw_1))
print("Sheet 2 rows retained:", len(raw_2_nonoverlap))
print("Combined rows:", len(combined))

Sheet 1 rows: 525461
Sheet 2 rows retained: 519387
Combined rows: 1044848


In [34]:

from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent
xlsx_path = project_dir / "data" / "online_retail_II.xlsx"

s1 = pd.read_excel(xlsx_path, sheet_name="Year 2009-2010")
s2 = pd.read_excel(xlsx_path, sheet_name="Year 2010-2011")

columns = list(s1.columns)
start = pd.Timestamp("2010-12-01")
end = pd.Timestamp("2010-12-09 23:59:59")

o1 = s1[pd.to_datetime(s1["InvoiceDate"]).between(start, end)]
o2 = s2[pd.to_datetime(s2["InvoiceDate"]).between(start, end)]

def row_counts(frame):
    normalized = frame[columns].copy()
    for col in columns:
        normalized[col] = (
            normalized[col].astype("string").fillna("<MISSING>")
        )
    return pd.util.hash_pandas_object(
        normalized, index=False
    ).value_counts().sort_index()

c1 = row_counts(o1)
c2 = row_counts(o2)

print("Sheet 1 overlap rows:", len(o1))
print("Sheet 2 overlap rows:", len(o2))
print("Matching row multiplicities:", c1.equals(c2))
print("Rows to exclude from sheet 2 if confirmed:", len(o2))


Sheet 1 overlap rows: 22523
Sheet 2 overlap rows: 22523
Matching row multiplicities: True
Rows to exclude from sheet 2 if confirmed: 22523


In [35]:

from pathlib import Path
import numpy as np
import pandas as pd

project_dir = Path.cwd().parent
xlsx_path = project_dir / "data" / "online_retail_II.xlsx"
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

raw_1 = pd.read_excel(xlsx_path, sheet_name="Year 2009-2010")
raw_2 = pd.read_excel(xlsx_path, sheet_name="Year 2010-2011")

# Exclude the verified overlap from the second worksheet.
dates_2 = pd.to_datetime(raw_2["InvoiceDate"])
overlap_start = pd.Timestamp("2010-12-01")
overlap_end = pd.Timestamp("2010-12-09 23:59:59")

raw_2 = raw_2.loc[
    ~dates_2.between(overlap_start, overlap_end)
].copy()

rename_map = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
}

df = pd.concat([raw_1, raw_2], ignore_index=True)
df = df.rename(columns=rename_map)

# Standardize text.
for col in ["invoice", "stock_code", "description", "country"]:
    df[col] = df[col].astype("string").str.strip()

# Standardize data types.
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["quantity"] = pd.to_numeric(df["quantity"], errors="raise")
df["price"] = pd.to_numeric(df["price"], errors="raise")
df["customer_id"] = pd.to_numeric(
    df["customer_id"], errors="coerce"
)

# Preserve the original eight fields for duplicate detection.
base_cols = [
    "invoice", "stock_code", "description", "quantity",
    "invoice_date", "price", "customer_id", "country"
]

df["line_value"] = df["quantity"] * df["price"]
df["is_cancellation_invoice"] = (
    df["invoice"].str.startswith("C", na=False)
)
df["missing_customer_id"] = df["customer_id"].isna()
df["negative_quantity"] = df["quantity"] < 0
df["zero_price"] = df["price"] == 0
df["negative_price"] = df["price"] < 0

df["is_in_duplicate_group"] = df.duplicated(
    subset=base_cols, keep=False
)
df["is_duplicate_extra"] = df.duplicated(
    subset=base_cols, keep="first"
)

df["transaction_category"] = np.select(
    [
        df["stock_code"].eq("B"),
        df["is_cancellation_invoice"],
        df["negative_quantity"],
    ],
    [
        "bad_debt_adjustment",
        "cancellation_style_invoice",
        "negative_quantity_non_c_invoice",
    ],
    default="other_or_ordinary",
)

df["review_status"] = np.select(
    [
        df["stock_code"].eq("B"),
        df["is_cancellation_invoice"],
        df["negative_quantity"],
        df["is_duplicate_extra"],
        df["missing_customer_id"],
    ],
    [
        "Review bad-debt adjustment",
        "Review cancellation-style invoice",
        "Review negative quantity",
        "Review duplicate occurrence",
        "Customer ID missing",
    ],
    default="No flagged condition",
)

csv_path = output_dir / "online_retail_prepared.csv"
df.to_csv(csv_path, index=False)

print("Prepared dataset shape:", df.shape)
print("Date range:", df["invoice_date"].min(), "to",
      df["invoice_date"].max())
print("CSV saved:", csv_path)
print("CSV size (MB):", round(csv_path.stat().st_size / 1_000_000, 2))
print("Duplicate extra occurrences:", int(df["is_duplicate_extra"].sum()))


Prepared dataset shape: (1044848, 18)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
CSV saved: D:\online-retail-analytics\data\processed\online_retail_prepared.csv
CSV size (MB): 185.89
Duplicate extra occurrences: 11812


In [36]:

from pathlib import Path
import numpy as np
import pandas as pd

project_dir = Path.cwd().parent
xlsx_path = project_dir / "data" / "online_retail_II.xlsx"
output_dir = project_dir / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

raw_1 = pd.read_excel(xlsx_path, sheet_name="Year 2009-2010")
raw_2 = pd.read_excel(xlsx_path, sheet_name="Year 2010-2011")

# Exclude the verified overlap from the second worksheet.
dates_2 = pd.to_datetime(raw_2["InvoiceDate"])
overlap_start = pd.Timestamp("2010-12-01")
overlap_end = pd.Timestamp("2010-12-09 23:59:59")

raw_2 = raw_2.loc[
    ~dates_2.between(overlap_start, overlap_end)
].copy()

rename_map = {
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "price",
    "Customer ID": "customer_id",
    "Country": "country",
}

df = pd.concat([raw_1, raw_2], ignore_index=True)
df = df.rename(columns=rename_map)

# Standardize text.
for col in ["invoice", "stock_code", "description", "country"]:
    df[col] = df[col].astype("string").str.strip()

# Standardize data types.
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
df["quantity"] = pd.to_numeric(df["quantity"], errors="raise")
df["price"] = pd.to_numeric(df["price"], errors="raise")
df["customer_id"] = pd.to_numeric(
    df["customer_id"], errors="coerce"
)

# Preserve the original eight fields for duplicate detection.
base_cols = [
    "invoice", "stock_code", "description", "quantity",
    "invoice_date", "price", "customer_id", "country"
]

df["line_value"] = df["quantity"] * df["price"]
df["is_cancellation_invoice"] = (
    df["invoice"].str.startswith("C", na=False)
)
df["missing_customer_id"] = df["customer_id"].isna()
df["negative_quantity"] = df["quantity"] < 0
df["zero_price"] = df["price"] == 0
df["negative_price"] = df["price"] < 0

df["is_in_duplicate_group"] = df.duplicated(
    subset=base_cols, keep=False
)
df["is_duplicate_extra"] = df.duplicated(
    subset=base_cols, keep="first"
)

df["transaction_category"] = np.select(
    [
        df["stock_code"].eq("B"),
        df["is_cancellation_invoice"],
        df["negative_quantity"],
    ],
    [
        "bad_debt_adjustment",
        "cancellation_style_invoice",
        "negative_quantity_non_c_invoice",
    ],
    default="other_or_ordinary",
)

df["review_status"] = np.select(
    [
        df["stock_code"].eq("B"),
        df["is_cancellation_invoice"],
        df["negative_quantity"],
        df["is_duplicate_extra"],
        df["missing_customer_id"],
    ],
    [
        "Review bad-debt adjustment",
        "Review cancellation-style invoice",
        "Review negative quantity",
        "Review duplicate occurrence",
        "Customer ID missing",
    ],
    default="No flagged condition",
)

csv_path = output_dir / "online_retail_prepared.csv"
df.to_csv(csv_path, index=False)

print("Prepared dataset shape:", df.shape)
print("Date range:", df["invoice_date"].min(), "to",
      df["invoice_date"].max())
print("CSV saved:", csv_path)
print("CSV size (MB):", round(csv_path.stat().st_size / 1_000_000, 2))
print("Duplicate extra occurrences:", int(df["is_duplicate_extra"].sum()))


Prepared dataset shape: (1044848, 18)
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
CSV saved: D:\online-retail-analytics\data\processed\online_retail_prepared.csv
CSV size (MB): 185.89
Duplicate extra occurrences: 11812


In [37]:

check = pd.read_csv(
    csv_path,
    usecols=["invoice", "invoice_date", "line_value",
             "transaction_category"]
)

print("CSV rows:", len(check))
print("Missing dates:", check["invoice_date"].isna().sum())
print("Missing line values:", check["line_value"].isna().sum())
print(check["transaction_category"].value_counts())


CSV rows: 1044848
Missing dates: 0
Missing line values: 0
transaction_category
other_or_ordinary                  1022284
cancellation_style_invoice           19165
negative_quantity_non_c_invoice       3393
bad_debt_adjustment                      6
Name: count, dtype: int64


In [38]:

import sqlite3

db_path = output_dir / "online_retail.db"

with sqlite3.connect(db_path) as conn:
    first_chunk = True

    for chunk in pd.read_csv(csv_path, chunksize=100_000):
        chunk.to_sql(
            "online_retail",
            conn,
            if_exists="replace" if first_chunk else "append",
            index=False,
        )
        first_chunk = False

    db_rows = conn.execute(
        "SELECT COUNT(*) FROM online_retail"
    ).fetchone()[0]

print("CSV rows:", len(df))
print("SQLite rows:", db_rows)
print("Counts match:", len(df) == db_rows)


CSV rows: 1044848
SQLite rows: 1044848
Counts match: True


In [39]:
from pathlib import Path
import pandas as pd
import sqlite3

project_dir = Path.cwd().parent
csv_path = project_dir / "data" / "processed" / "online_retail_prepared.csv"
db_path = project_dir / "data" / "processed" / "online_retail.db"

df = pd.read_csv(csv_path, parse_dates=["invoice_date"])

with sqlite3.connect(db_path) as conn:
    print(pd.read_sql_query(
        "SELECT COUNT(*) AS row_count FROM online_retail",
        conn
    ))

   row_count
0    1044848
